In [2]:
import torch
import pandas as pd
import sys
from pathlib import Path
from torch_geometric.loader import DataLoader
sys.path.append(str(Path.cwd().parent))
from config import HARMONIZED_DATA_PATH
from gnn_models_utils import (
    smiles_to_graph,
    GCNDrugEncoder,
    GATDrugEncoder,
    GINDrugEncoder
)

device = torch.device("cuda" if torch.cuda.is_available() else "mps")
node_feature_dim = 4
hidden_dim = 64
batch_size = 10

In [3]:
df = pd.read_parquet(HARMONIZED_DATA_PATH)

In [18]:
def verify_forward_pass(model, batch, expected_hidden_dim, model_name):
    model = model.to(device)
    model.train()  # dropout included
    
    try:
        out = model(batch.x.float(), batch.edge_index)        
        expected_shape = (batch.x.shape[0], expected_hidden_dim)
        assert out.shape == expected_shape, f"Shape error! Expected {expected_shape}, got {out.shape}"
        assert not torch.isnan(out).any(), "Error: The output contains NaN values!"
        assert not torch.isinf(out).any(), "Error: The output contains Inf values!"
        
        print(f"[OK] {model_name}: Forward pass successful!")
        print(f"     Output Shape: {out.shape} | Min: {out.min().item():.4f} | Max: {out.max().item():.4f}\n")
        
    except Exception as e:
        print(f"[ERROR] {model_name} failed: {e}\n")

In [19]:
target = 'LN_IC50'
smiles_raw = df['SMILES'].values
X_genomic_raw = df.filter(regex=r'.* \(.*\)').values
y_raw = df[target].values
drug_groups = df['DRUG_ID'].values

real_graphs = []
for s in smiles_raw[:10]:
    g = smiles_to_graph(s)
    if g is not None:
        real_graphs.append(g)

real_loader = DataLoader(real_graphs, batch_size=10, shuffle=False)
batch = next(iter(real_loader)).to(device)
node_feature_dim = batch.x.shape[1]
hidden_dim = 64

gcn_model = GCNDrugEncoder(node_feature_dim=node_feature_dim, hidden_dim=hidden_dim, dropout=0.1)
gin_model = GINDrugEncoder(node_feature_dim=node_feature_dim, hidden_dim=hidden_dim, dropout=0.1)

verify_forward_pass(gcn_model, batch, hidden_dim, "GCNDrugEncoder (Real data)")
verify_forward_pass(gin_model, batch, hidden_dim, "GINDrugEncoder (Real data)")

print(f"Test batch created on{device}:")
print(f"  - Number of graphs in the batch: {batch.num_graphs}")
print(f"  - Total number of atoms (nodes): {batch.x.shape[0]}")
print(f"  - Input Shape (x): {batch.x.shape}")
print(f"  - Edge Index Shape: {batch.edge_index.shape}\n")

[OK] GCNDrugEncoder (Real data): Forward pass successful!
     Output Shape: torch.Size([246, 64]) | Min: -4.8976 | Max: 3.2121

[OK] GINDrugEncoder (Real data): Forward pass successful!
     Output Shape: torch.Size([246, 64]) | Min: -6.2226 | Max: 7.2498

Test batch created onmps:
  - Number of graphs in the batch: 10
  - Total number of atoms (nodes): 246
  - Input Shape (x): torch.Size([246, 2])
  - Edge Index Shape: torch.Size([2, 516])

